Обзор архитектур, лежащих в основе современных сервисов вроде Suno, Udio, MusicGen, Stable Audio и других.

---

## Оглавление

1. [Почему генерация музыки — особенная задача](#1-почему-генерация-музыки--особенная-задача)
2. [Краткая история: от символьной музыки к аудио](#2-краткая-история-от-символьной-музыки-к-аудио)
3. [Базовые представления аудио в ML](#3-базовые-представления-аудио-в-ml)
4. [Нейросетевые аудио-кодеки: фундамент современной генерации](#4-нейросетевые-аудио-кодеки-фундамент-современной-генерации)
5. [Архитектурная семья №1: авторегрессионные трансформеры над токенами](#5-архитектурная-семья-1-авторегрессионные-трансформеры-над-токенами)
6. [Архитектурная семья №2: латентная диффузия](#6-архитектурная-семья-2-латентная-диффузия)
7. [Условная генерация: текст, мелодия, стиль](#7-условная-генерация-текст-мелодия-стиль)
8. [Что (предположительно) под капотом у Suno и Udio](#8-что-предположительно-под-капотом-у-suno-и-udio)
9. [Сравнительная таблица моделей](#9-сравнительная-таблица-моделей)
10. [Метрики качества и оценка](#10-метрики-качества-и-оценка)
11. [Открытые проблемы и направления развития](#11-открытые-проблемы-и-направления-развития)
12. [Что читать дальше](#12-что-читать-дальше)

---

## 1. Почему генерация музыки — особенная задача

Музыка — это не просто длинный временной ряд. У неё есть несколько свойств, которые делают её сложнее как изображений, так и текста:

- **Огромная размерность.** Стерео-аудио 44.1 кГц — это 88 200 чисел в секунду. Трёхминутная песня — около 16 миллионов сэмплов. Никакой трансформер не работает с такими последовательностями напрямую.
- **Многомасштабная структура.** На микроуровне — фазовые отношения между сэмплами (миллисекунды), на мезоуровне — ритм и гармония (секунды), на макроуровне — куплет/припев/бридж (минуты). Хорошая модель должна согласованно работать на всех трёх масштабах одновременно.
- **Полифония и одновременность.** В отличие от речи, где в каждый момент звучит один голос, в музыке параллельно идут вокал, барабаны, бас, гармония — и они должны быть согласованы ритмически и гармонически.
- **Восприятие сильно нелинейно.** Человек прощает шум в высоких частотах, но мгновенно слышит фальшивую ноту или сбой ритма. Поэтому L2-loss на сырых сэмплах работает плохо — нужны перцептуальные представления.
- **Вокал + инструменты + текст одновременно.** Полноценная песня требует синтеза разборчивого вокала с правильной артикуляцией, который при этом ритмически и гармонически связан с аккомпанементом.

Эти ограничения и определяют все архитектурные решения, которые мы рассмотрим дальше.

---

## 2. Краткая история: от символьной музыки к аудио

Генерация музыки прошла четыре эпохи.

**Эпоха символьной музыки (до ~2018).** Модели работали с MIDI — последовательностью нот, длительностей и громкостей. Это удобно (низкая размерность), но результат нельзя «услышать» без отдельного синтезатора, и тембр, нюансы исполнения, шумы — всё это теряется. Сюда относятся ранние LSTM-модели, Music Transformer от Google (2018), MuseNet от OpenAI.

**Эпоха автокодировщиков сырого аудио (~2018–2020).** WaveNet и Jukebox от OpenAI пытались моделировать сэмплы напрямую. Jukebox, например, использовал иерархический VQ-VAE, чтобы сжать аудио в три уровня абстракции, и трансформеры — для генерации в каждом уровне. Результаты звучали узнаваемо, но «мутно», и генерация одной песни занимала часы.

**Эпоха нейронных кодеков и латентных моделей (2021–2023).** Прорыв пришёл с моделями SoundStream (Google, 2021) и EnCodec (Meta, 2022) — нейронными кодеками, которые сжимают аудио в дискретные токены с потерями, но сохраняя качество. Это позволило применить к аудио всю инфраструктуру языкового моделирования: MusicLM, MusicGen, AudioLM.

**Эпоха коммерческих text-to-song моделей (2023 — настоящее).** Suno (декабрь 2023), Udio (2024), Stable Audio. Главное отличие — синтез связного вокала с осмысленным текстом плюс полная аранжировка, всё из текстового промпта.

---

## 3. Базовые представления аудио в ML

Прежде чем говорить про модели, нужно понять, в каком виде они «видят» музыку.

### 3.1. Waveform — сырая форма волны

Просто массив чисел: амплитуда в каждый момент времени. Это «правда», но размерность чудовищная, и модель должна сама выучивать понятия «частота», «гармоника», «такт» — это очень неэффективно.

### 3.2. Спектрограмма

Применяем оконное преобразование Фурье (STFT) — получаем 2D-картинку: ось X — время, ось Y — частота, цвет — энергия. Спектрограмма гораздо ближе к тому, как звук воспринимает ухо, и её можно обрабатывать свёрточными сетями как обычное изображение. На этом основан, например, Riffusion — буквально fine-tune Stable Diffusion на спектрограммах.

Подвид — **мел-спектрограмма**, где ось частот сжата по мел-шкале (логарифмическая, имитирует восприятие). Это стандарт в речевых моделях.

Минус спектрограмм: при обратном переходе в звук (через алгоритм Griffin-Lim или нейронный вокодер) теряется фаза, и качество страдает. Поэтому современные модели редко работают с мел-спектрограммами напрямую.

### 3.3. Латентные представления и дискретные токены

Главная идея современной генерации: обучить **отдельную модель-кодек**, которая сжимает аудио в компактное представление, и потом учить генеративную модель уже на этом представлении. Об этом — следующий раздел.

---

## 4. Нейросетевые аудио-кодеки: фундамент современной генерации

Это, пожалуй, ключевая концепция всей области. Почти все современные модели (MusicGen, MusicLM, Suno, Stable Audio) опираются на нейросетевой кодек.

### 4.1. Идея

Учим энкодер-декодер: энкодер сжимает аудио в маленькое представление, декодер восстанавливает аудио обратно. Если сжатие достаточно агрессивное и квантизованное (т.е. представление — это последовательность целых чисел из конечного словаря, «токенов»), мы получаем для аудио то же самое, что BPE-токенизация даёт для текста: возможность работать языковыми моделями.

### 4.2. SoundStream (Google, 2021) и EnCodec (Meta, 2022)

Это два главных кодека. Архитектурно они близки:

- **Энкодер.** Свёрточная сеть, которая берёт waveform (например, 24 кГц) и понижает частоту дискретизации в ~320 раз. На выходе — последовательность векторов размерности D с частотой ~75 Гц.
- **Квантизатор.** Каждый вектор приближается ближайшим из обученного словаря (codebook). Чтобы повысить точность, используется **Residual Vector Quantization (RVQ)**: первый квантизатор кодирует вектор грубо, второй — кодирует ошибку первого, третий — ошибку второго, и так далее.
- **Декодер.** Зеркальная свёрточная сеть, восстанавливает waveform.
- **Обучение.** Комбинация перцептуальной L1/L2-loss, спектральной loss и **GAN-дискриминаторов** (как в обычной генерации изображений), чтобы аудио звучало реалистично.

### 4.3. Зачем нужна RVQ

Простая идея: чтобы закодировать вектор одним токеном из словаря 1024, нужен codebook на 1024 точки — для качественного аудио этого мало. Если же использовать **8 уровней по 1024 токена каждый** (где каждый последующий кодирует ошибку предыдущего), мы получаем эффективный словарь 1024⁸ ≈ 10²⁴ комбинаций при гораздо меньшей сложности обучения. Это и есть RVQ.

В результате одну секунду аудио (24 кГц) можно представить как **матрицу токенов размером 75 × 8** (75 шагов времени, 8 уровней RVQ). Это уже работоспособный масштаб для трансформеров.

### 4.4. Почему это так важно

Кодек решает сразу несколько проблем:

- Снижает размерность в сотни раз — генерация становится вычислительно осуществимой.
- Даёт дискретные токены — можно прямо применять next-token prediction, как в LLM.
- Декодер обучен синтезировать «реалистично звучащее» аудио — модель-генератор не должна сама учиться синтезу из сэмплов.

С этим инструментом в руках можно строить генеративные модели двумя принципиально разными путями: авторегрессией над токенами или диффузией в латентном пространстве. К ним и переходим.

---

## 5. Архитектурная семья №1: авторегрессионные трансформеры над токенами

Это путь, выбранный MusicLM (Google) и MusicGen (Meta). Идея максимально близка к языковым моделям: раз аудио — это последовательность токенов, давайте обучать GPT-подобную модель предсказывать следующий токен.

### 5.1. MusicLM (Google, январь 2023)

Иерархический подход с тремя уровнями токенов:

1. **Семантические токены** (MuLan + w2v-BERT) — высокоуровневое описание «о чём» музыка. Несут информацию о жанре, инструментах, общем настроении, медленно меняются во времени.
2. **Грубые акустические токены** (первые уровни RVQ SoundStream) — общая форма звука.
3. **Тонкие акустические токены** (последние уровни RVQ) — детали тембра и нюансы.

Генерация — последовательно: текст → семантические → грубые акустические → тонкие акустические → декодер SoundStream → waveform. Три отдельных трансформера. Качество хорошее, но архитектура сложная и медленная.

### 5.2. MusicGen (Meta, июнь 2023)

Главное упрощение: всё делается **одним трансформером**, работающим напрямую с токенами EnCodec.

Проблема в том, что у нас 4 (или 8) параллельных потоков токенов RVQ на каждом шаге времени. Если их сериализовать наивно («сначала все токены первого уровня, потом второго...»), последовательность будет слишком длинной. MusicGen предложил несколько паттернов чередования, среди которых **delay pattern**: каждый уровень RVQ сдвинут относительно предыдущего на один шаг. Это позволяет за один проход декодера сгенерировать все 4 уровня одновременно, теряя минимум информации о причинности.

Преимущества MusicGen:

- Один трансформер вместо трёх.
- Открытые веса (300M / 1.5B / 3.3B параметров).
- На один шаг по времени — генерация всех уровней одновременно.

Условие на текст подаётся через cross-attention: текст кодируется T5-encoder, и трансформер «смотрит» на это представление при генерации.

### 5.3. AudioLM и общая схема

Более общий шаблон, к которому относятся все авторегрессионные модели:

```
текст ──► текстовый энкодер (T5/CLAP/MuLan) ──► условные эмбеддинги
                                                       │
                                                       ▼
                                       ┌──── Трансформер-декодер ────┐
                                       │  предсказывает токены       │
                                       │  кодека авторегрессионно    │
                                       └──────────────┬──────────────┘
                                                      ▼
                                            токены аудио-кодека
                                                      │
                                                      ▼
                                       декодер кодека (EnCodec/SoundStream)
                                                      │
                                                      ▼
                                                  waveform
```

### 5.4. Плюсы и минусы авторегрессии

**Плюсы:** естественно работает с переменной длиной, можно стримить генерацию, легко делать продолжения и заполнение пропусков (inpainting через специальные маски).

**Минусы:** медленно (генерируем токен за токеном), accumulation of errors (одна неудачная нота может «увести» всю генерацию), труднее обеспечить глобальную согласованность.

---

## 6. Архитектурная семья №2: латентная диффузия

Параллельный путь, который занял Stability AI (Stable Audio), AudioLDM, MusicLDM, и который, по общему мнению, частично использует Suno.

### 6.1. Идея диффузии в одну минуту

Берём чистый сэмпл данных x₀. Постепенно добавляем гауссов шум, получая x₁, x₂, ..., x_T — на шаге T это уже чистый шум. Учим нейросеть на каждом шаге **предсказывать добавленный шум** (или, эквивалентно, чуть менее зашумлённую версию). На инференсе стартуем с шума и итеративно очищаем его, прогоняя модель T раз. На каждом шаге можно подавать условие (текст, эмбеддинг) — это и даёт текстовое управление.

### 6.2. Почему латентная

Делать диффузию напрямую на спектрограмме или waveform дорого. Решение: сначала сжать аудио автокодировщиком (VAE) в латентное пространство, делать диффузию там, потом декодировать обратно. Так работают и Stable Diffusion (для картинок), и большинство аудио-моделей.

### 6.3. AudioLDM и MusicLDM

AudioLDM (2023) — первая успешная адаптация Latent Diffusion к аудио. Архитектура:

1. **VAE-энкодер** сжимает мел-спектрограмму в латент.
2. **U-Net** с cross-attention делает диффузию в латентном пространстве, условясь на CLAP-эмбеддинге текста (CLAP — это аудио-аналог CLIP, обученный сопоставлять описания и звуки).
3. **VAE-декодер** восстанавливает мел-спектрограмму.
4. **Вокодер HiFi-GAN** превращает мел-спектрограмму в waveform.

MusicLDM — это специализированная для музыки версия AudioLDM с переобученным CLAP на музыкальных данных и хитрыми augmentations на основе beat-tracking (для синхронного по битам микса).

### 6.4. Stable Audio (Stability AI, 2023–2024)

Дальнейшее развитие линии. Ключевые отличия:

- **Стерео 44.1 кГц.** Не моно и не 16 кГц, а настоящее музыкальное качество.
- **DiT вместо U-Net.** Diffusion Transformer — трансформер, работающий с латентами как с последовательностью токенов; масштабируется лучше U-Net.
- **Кондиционирование на длительность.** Модели подаётся целевая длина трека — это решает проблему «диффузия всегда выдаёт фиксированную длину».
- **Обучена только на лицензированных данных** (по утверждению Stability), что важно с юридической точки зрения.

### 6.5. Плюсы и минусы диффузии

**Плюсы:** высокое аудио-качество, хорошая глобальная согласованность (модель «видит» всю композицию сразу), естественная поддержка inpainting/editing через guidance, параллельная генерация.

**Минусы:** фиксированная длина (нужны трюки для переменной), медленнее на один прогон (хотя меньше прогонов, чем токенов в авторегрессии), труднее стримить, сложнее с длинными композициями (несколько минут).

---

## 7. Условная генерация: текст, мелодия, стиль

Главная фишка современных сервисов — управление промптом. Как именно текст влияет на генерацию?

### 7.1. Текстовые энкодеры

Несколько вариантов:

- **T5** (Google) — общий текстовый трансформер. Используется в MusicGen. Просто кодирует промпт в последовательность эмбеддингов.
- **CLAP** (Contrastive Language-Audio Pretraining) — модель, обученная contrastive-loss'ом сопоставлять описания и аудио (как CLIP для картинок). Используется в AudioLDM. Преимущество: эмбеддинги уже «знают» про звук.
- **MuLan** (Google) — аналог CLAP, использовался в MusicLM.

Текстовое представление подаётся в генератор через cross-attention или просто конкатенацией к последовательности.

### 7.2. Кондиционирование на мелодии и аудио-референсе

MusicGen умеет «продолжать стиль» референсной мелодии — для этого мелодия извлекается алгоритмически (chromagram), и хроматические признаки подаются как дополнительное условие. Suno и Udio позволяют загрузить референсный аудио-фрагмент и продолжить его — это работает либо через extension (модель просто продолжает с того места), либо через style transfer на эмбеддингах.

### 7.3. Структурные теги (для песен)

Suno поддерживает разметку в тексте: `[Verse]`, `[Chorus]`, `[Bridge]`, `[Outro]`. Модель училась на песнях с такой разметкой и научилась переключать «режим» — в припеве появляется развёрнутая аранжировка, в куплете — более минималистичный звук. Это даёт пользователю контроль над структурой без необходимости рисовать MIDI.

### 7.4. Кондиционирование на текст песни

Самая сложная часть — синтез вокала, который **поёт конкретные слова**. Здесь нужна тесная связь между:

1. Лингвистическим представлением текста (фонемы, ударения).
2. Мелодической линией (какая нота — на какой слог).
3. Тембром голоса и стилем исполнения.

Это де-факто отдельная модель внутри пайплайна, родственная TTS (text-to-speech), но с гораздо большим диапазоном экспрессии. О том, как это делает Suno, информации мало, но почти наверняка используется отдельный модуль выравнивания текста и мелодии (вроде того, что в Bark — открытой модели от Suno для речи и пения).

---

## 8. Что (предположительно) под капотом у Suno и Udio

Точные архитектуры этих сервисов не публиковались, но из обрывочных деталей и наблюдаемого поведения можно реконструировать общую схему.

### 8.1. Suno

Известно (из официальных источников и анализа):

- **Bark** — открытая модель Suno для генерации голоса и пения, основанная на токенах и трансформере. Скорее всего, эволюционная линия для вокала в коммерческом продукте.
- **Chirp** — внутреннее название инструментальной части генерации.
- В Suno v3 и v4 наблюдалась характерная для авторегрессионных моделей деградация на длинных треках (артефакты накапливались), что косвенно указывает на гибридную или авторегрессионную архитектуру в этой части.
- В Suno v5 (сентябрь 2025) длина связной генерации выросла до ~8 минут с заметным улучшением вокала — вероятно, переход на более совершенную архитектуру с лучшей глобальной согласованностью (возможно, гибрид планировщика и диффузионного синтезатора).

Наиболее правдоподобная схема Suno:

```
текст промпта + lyrics
       │
       ▼
  ┌────────────────────────────────┐
  │ Планировщик (LLM-подобный):    │
  │ генерирует структуру песни,    │
  │ выбор стиля, мелодическую      │
  │ канву, аранжировочные решения  │
  └────────────────┬───────────────┘
                   │
                   ▼
  ┌──────────────────────────────────────┐
  │ Вокальный синтез (Bark-наследник):   │
  │ выравнивание текст ↔ мелодия,        │
  │ генерация вокальных токенов          │
  └────────────────┬─────────────────────┘
                   │
                   ▼
  ┌──────────────────────────────────────┐
  │ Инструментальный синтез:             │
  │ генерация остальных треков           │
  │ (авторегрессия или диффузия)         │
  └────────────────┬─────────────────────┘
                   │
                   ▼
            нейросетевой декодер
                   │
                   ▼
              стерео-аудио
```

### 8.2. Udio

Udio (основан выходцами из Google DeepMind) показывает заметно более чистое аудио на инструментальной части и более «бесшовные» переходы — это часто связывают с большей долей диффузии в их пайплайне. Технических деталей открыто очень мало.

### 8.3. Общий вывод

И Suno, и Udio — это **не одна модель**, а пайплайн из нескольких специализированных, передающих друг другу промежуточные представления. Это очень похоже на то, как устроены современные TTS-системы (планировщик → акустическая модель → вокодер), только с гораздо более богатыми возможностями условного управления.

---

## 9. Сравнительная таблица моделей

| Модель | Год | Тип | Открытая? | Кодек / латент | Текст-энкодер | Особенности |
|---|---|---|---|---|---|---|
| Jukebox | 2020 | AR + иерархический VQ-VAE | ✅ | свой VQ-VAE | — | Первая успешная end-to-end |
| Riffusion | 2022 | Diffusion на спектрограммах | ✅ | — | CLIP | Fine-tune Stable Diffusion |
| MusicLM | 2023 | AR, иерархический трансформер | ❌ | SoundStream | MuLan | 3 уровня токенов |
| MusicGen | 2023 | AR, один трансформер | ✅ | EnCodec | T5 | Delay pattern для RVQ |
| AudioLDM / AudioLDM2 | 2023 | Латентная диффузия | ✅ | VAE + HiFi-GAN | CLAP / FLAN-T5 | Универсальный аудио-генератор |
| MusicLDM | 2023 | Латентная диффузия | ✅ | VAE + HiFi-GAN | CLAP (music) | Beat-synchronous mixup |
| Stable Audio 1/2 | 2023–24 | Латентная диффузия (DiT) | частично | свой VAE | T5 | Стерео 44.1 кГц, кондиц. на длину |
| Suno (v3–v5) | 2023–25 | Гибрид (вероятно) | ❌ | свой | свой | Вокал + lyrics + структура |
| Udio | 2024 | Вероятно гибрид | ❌ | свой | свой | Высокое инструментальное качество |

---

## 10. Метрики качества и оценка

Оценить генерацию музыки объективно сложно — это во многом субъективная задача. Тем не менее используются несколько метрик.

### 10.1. Автоматические метрики

- **FAD (Fréchet Audio Distance).** Аналог FID из computer vision. Берём предобученную сеть (обычно VGGish), извлекаем эмбеддинги для реальных и сгенерированных аудио, считаем дистанцию между распределениями. Чем меньше — тем ближе генерация к «реальной» музыке.
- **KL-дивергенция на эмбеддингах.** Похожая идея: сравниваем распределения признаков.
- **CLAP-score.** Считаем косинусную близость между CLAP-эмбеддингом текстового промпта и CLAP-эмбеддингом сгенерированного аудио. Метрика text-audio alignment — насколько генерация соответствует промпту.
- **Inception Score** и его аудио-аналоги — измеряют разнообразие и узнаваемость.

### 10.2. Субъективная оценка

Никакая автоматическая метрика не заменяет слуховой тест. Стандартная процедура — **MUSHRA** или просто mean opinion score (MOS) по нескольким осям:

- Общее качество звука.
- Соответствие промпту.
- Музыкальность (гармония, ритм, структура).
- Приятность на слух (preference).

Современные лидерборды (например, на TTS-Arena-Music, аналоги Chatbot Arena для аудио) опираются именно на парные предпочтения слушателей.

### 10.3. Что метрики не ловят

- Долговременную согласованность (метрики обычно считают на 10-секундных отрезках).
- Качество вокала и разборчивость лирики — для этого нужны отдельные ASR-based метрики.
- Оригинальность vs. плагиат тренировочных треков.

---

## 11. Открытые проблемы и направления развития

**Длительность и согласованность.** Связная генерация дольше 3–5 минут с правильной структурой (куплет/припев) и единым тембром — до сих пор сложно. Suno v5 претендует на 8 минут, но при внимательном прослушивании заметны деградации.

**Контроль и редактирование.** Большинство моделей — «чёрный ящик»: задаёшь промпт, получаешь трек. Точечное редактирование (поменять только бас, перепеть один куплет, изменить эмоциональную окраску бриджа) — активная область исследований. Inpainting через диффузию работает, но грубо.

**Юридические и этические вопросы.** Большинство сильных моделей обучены на нелицензированных каталогах. В 2024 году major-лейблы подали иски против Suno и Udio. Это толкает индустрию к моделям, обученным на лицензированных датасетах (Stable Audio, MusicGen), что пока даёт более скромное качество.

**Real-time и интерактивность.** Существующие модели плохо работают в реальном времени (latency секунды), но для live-импровизации, музыкальных игр и интерактивных приложений нужны модели с задержкой <100 мс. Это требует пересмотра архитектур — скорее всего, в сторону streaming-моделей.

**Multimodal music.** Видео-аккомпанемент (генерация музыки под видео), генерация по эмоциональным сигналам, по биометрии — все это активно исследуется.

**MIDI-аудио гибриды.** Сильное направление — модели, которые внутри оперируют структурой (нотами, аккордами), а наружу выдают аудио. Это даёт и качество, и редактируемость. Примеры — Anticipatory Music Transformer, MIDI-AudioLDM.

---

## 12. Что читать дальше

**Базовые статьи (с порядком чтения):**

1. *Neural Discrete Representation Learning* (van den Oord et al., 2017) — VQ-VAE, основа всех нейронных кодеков.
2. *SoundStream: An End-to-End Neural Audio Codec* (Zeghidour et al., 2021) — RVQ и нейронные кодеки.
3. *High Fidelity Neural Audio Compression* (Défossez et al., 2022) — EnCodec.
4. *AudioLM* (Borsos et al., 2022) — общая схема языкового моделирования аудио.
5. *MusicLM* (Agostinelli et al., 2023) — иерархическая генерация музыки.
6. *Simple and Controllable Music Generation* (Copet et al., 2023) — MusicGen.
7. *AudioLDM* (Liu et al., 2023) — латентная диффузия для аудио.
8. *Stable Audio 2.0* technical report — современный стандарт open-weight генерации.

**Полезные репозитории:**

- `facebookresearch/audiocraft` — MusicGen, EnCodec, AudioGen.
- `Stability-AI/stable-audio-tools` — Stable Audio.
- `haoheliu/AudioLDM` — AudioLDM.
- `suno-ai/bark` — Bark (открытая модель Suno для речи и пения).

**Курсы и обзоры:**

- Лекции CCRMA (Stanford) по DSP — фундамент, без которого трудно понимать обработку аудио.
- Обзоры на сайтах Hugging Face и Papers With Code, секция Audio Generation.

---

*Документ написан в мае 2026 года. Область развивается очень быстро — конкретные модели и их версии устаревают, но базовые архитектурные принципы (нейронные кодеки, авторегрессия по токенам, латентная диффузия, гибридные пайплайны) останутся релевантны как минимум на несколько лет.*